<a href="https://colab.research.google.com/github/pizzeman/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [120]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [121]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
orders = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
assert len(orders) == 4
assert orders['qty'].sum() == 7
orders

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [122]:
# TODO
result = orders.merge(pd.json_normalize(vendors_json), how='left', indicator=True)

print("No Match: ", result.loc[result['name'].isna()])
print("Units: ", result['qty'].sum())

No Match:        item  qty order vendor_id name zone     _merge
3  Hot Dog    3     3      V-07  NaN  NaN  left_only
Units:  7


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [123]:
# TODO
result['name'] = result['name'].astype('object').fillna('Unmatched')
result.groupby('name')['qty'].sum()
# I am deciding to keep the unmatched and just saying it is unmatched. The unit sold is high (3 units), so clearly this data is important, with just missing data

,qty
name,
Cav Merch,1
Hoos Burgers,3
Unmatched,3


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [124]:
# TODO
result['zone'] = result['zone'].astype('object').fillna('Unmatched')
result.groupby('zone')['qty'].sum()

# The total is 7 and the zone with the most items is A with 4

,qty
zone,
A,4
Unmatched,3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [125]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

df = pd.DataFrame(ragged_json)

df['lines'] = df['lines'].apply(lambda x: x if isinstance(x, list) else [])

exploded = df.explode('lines')

lines_df = pd.json_normalize(exploded['lines'].dropna())

df_flat = (
    df[['order', 'vendor_id']]
    .reset_index(drop=True)
    .join(lines_df)
    .fillna('Missing')
)

print(df_flat)


   order vendor_id     item      qty
0      4      V-01     Soda      2.0
1      5      V-10    Fries  Missing
2      6      V-01  Missing  Missing


### Q6 — Validate

In [126]:
assert len(orders) == 4
assert orders['qty'].sum() == 7
assert len(result) == len(orders), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a. Per order would have a lot more rows than per line item. Per line item aggregates items with multiple quantities. The total quantity is easier to calcualte and identifying with vendor sold the most.
b. Practically, a quantity of zero is included is calculations whereas a missing quantity is not. For example, an average would be influenced by the quantity of zero even if it is just meant to represent missing data rather than recording zero units.